In [7]:
#!pip install skrebate scikit-learn pandas numpy matplotlib seaborn

$\textbf{Step 1: Importing all necessary libraries}$

In [8]:
import pandas as pd
import numpy as np
import seaborn as sns
import warnings, time, pickle
warnings.filterwarnings('ignore')

#Importing The ReliefF Library to Implement for Phase 2
from skrebate import ReliefF

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score

#Importing Libraries for the 5 Classifiers that will be used
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

$\textbf{Step 1: Loading Datasets and Configurations}$
$\textbf{ReliefF Algorithm}$

In [9]:
DATASETS = {
    'Dataset1' : {'train': 'Datasets/1/train.csv','test': 'Datasets/1/test.csv'},
    'Dataset2' : {'train': 'Datasets/2/train.csv','test': 'Datasets/2/test.csv'},
    'Dataset3' : {'train': 'Datasets/3/train.csv','test': 'Datasets/3/test.csv'},
    'Dataset4' : {'train': 'Datasets/4/train.csv','test': 'Datasets/4/test.csv'},
    'Dataset5' : {'train': 'Datasets/5/train.csv','test': 'Datasets/5/test.csv'},
    'Dataset6' : {'train': 'Datasets/6/train.csv','test': 'Datasets/6/test.csv'},
    'Dataset7' : {'train': 'Datasets/7/train.csv','test': 'Datasets/7/test.csv'},
    'Dataset8' : {'train': 'Datasets/8/train.csv','test': 'Datasets/8/test.csv'},
    # 'Dataset9' : {'train': 'Datasets/9/train.csv','test': 'Datasets/9/test.csv'},
    # 'Dataset10' : {'train': 'Datasets/10/train.csv','test': 'Datasets/10/test.csv'},
    # 'Dataset11' : {'train': 'Datasets/11/train.csv','test': 'Datasets/11/test.csv'},
    # 'Dataset12' : {'train': 'Datasets/12/train.csv','test': 'Datasets/12/test.csv'},
    # 'Dataset13' : {'train': 'Datasets/13/train.csv','test': 'Datasets/13/test.csv'},
    # 'Dataset14' : {'train': 'Datasets/14/train.csv','test': 'Datasets/14/test.csv'},
    # 'Dataset15' : {'train': 'Datasets/15/train.csv','test': 'Datasets/15/test.csv'},
    # 'Dataset16' : {'train': 'Datasets/16/train.csv','test': 'Datasets/16/test.csv'},
}

LABEL_COL = 'Label'
N_FOLDS = 10
INNER_FOLDS = 3
RANDOM_STATE = 42
# For Datasets 1-8 
K_CANDIDATES = [5, 10, 15]
# For Datasets 9-16 
# K_CANDIDATES = [50, 100, 150] 
RELIEFF_N_NEIGHBORS = 10  

CHECKPOINT_FILE = 'results_checkpoint_1-8.pkl'

$\textbf{Step 2: Classifier Definition with Hyperparameter Grids}$

In [10]:
CLASSIFIERS = {
    'SVM': (
    SVC(random_state=RANDOM_STATE),
    {
        'classifier__C':      [1, 10],
        'classifier__kernel': ['rbf', 'linear'],
        'classifier__gamma':  ['scale', 'auto'],
    }
    ),
    'kNN': (
        KNeighborsClassifier(),
        {
            'classifier__n_neighbors': [3, 5, 11, 21],
            'classifier__weights':     ['uniform', 'distance'],
        }
    ),
    'Decision Tree': (
        DecisionTreeClassifier(random_state=RANDOM_STATE),
        {
            'classifier__max_depth':        [5, 10, 20, None],
            'classifier__min_samples_split': [2, 5, 10],
        }
    ),
    'Random Forest': (
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        {
            'classifier__n_estimators': [50, 100, 200],
            'classifier__max_depth':    [5, 10, None],
        }
    ),
    'MLP': (
        MLPClassifier(max_iter=500, random_state=RANDOM_STATE, early_stopping=True),
        {
            'classifier__hidden_layer_sizes': [(64,), (128,)],
            'classifier__alpha':              [1e-4, 1e-3],
            'classifier__learning_rate_init': [0.001, 0.01],
            'classifier__learning_rate':      ['constant', 'adaptive'],
        }
    ),
}

$\textbf{Step 4: Helper Functions}$

In [11]:
def load_dataset(path, label_col=LABEL_COL):
    df = pd.read_csv(path)
    X = df.drop(columns=[label_col]).values.astype(np.float64) #Dropping the Label Column and converts to float64 Numpy array
    y_raw = df[label_col].values
    y = LabelEncoder().fit_transform(y_raw) #Encoded Data for consiste
    print(f'  Loaded: {X.shape[0]} samples, {X.shape[1]} features, '
          f'{len(np.unique(y))} classes')
    return X, y

def apply_relieff(X_train, y_train, k):
    imputer = SimpleImputer(strategy='mean')
    X_relief_imp = imputer.fit_transform(X_train)

    selector = ReliefF(n_features_to_select=k, n_neighbors=RELIEFF_N_NEIGHBORS)
    selector.fit(X_relief_imp, y_train)
    top_idx = np.argsort(selector.feature_importances_)[::-1][:k]
    return X_train[:, top_idx], selector.feature_importances_, top_idx

def build_pipeline(clf):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler()),
        ('classifier', clf),
    ])

def evaluate_classifier(clf, param_grid, X_train, y_train, X_test, y_test):
    pipe = build_pipeline(clf)
    inner_cv = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True,
                                random_state = RANDOM_STATE)
    gs = GridSearchCV(pipe, param_grid, cv=inner_cv,
                      scoring='f1_macro', n_jobs=-1, refit=True)
    gs.fit(X_train, y_train)
    y_pred = gs.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division = 0)
    best_params = gs.best_params_
    return acc, f1, best_params

def print_fold_results(fold_idx, n_folds, ds_name, all_results, clf_names, best_k):
    print(f"\n  ── Fold {fold_idx + 1}/{n_folds} Results ({ds_name}) | best_k={best_k} ──")
    
    rows = []
    for phase_name, classifiers in all_results[ds_name].items():
        for clf_name, metrics in classifiers.items():
            if len(metrics['acc']) == 0:
                continue
            rows.append({
                'Phase':      'Baseline' if phase_name == 'baseline' else 'ReliefF',
                'Classifier': clf_name,
                'Accuracy':  f"{metrics['acc'][fold_idx]:.4f}",
                'Macro-F1':  f"{metrics['f1'][fold_idx]:.4f}",
                'Best Params': metrics['params'][-1],
            })
    
    df = pd.DataFrame(rows)
    pd.set_option('display.max_colwidth', 100)
    display(df)
    print()


$\textbf{Step 5: Main Experiment Loop}$

In [12]:
def training_models(datasets, classifiers):
    all_results    = {}
    relieff_scores = {}
    best_k_per_ds = {}
    outer_cv       = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                                     random_state=RANDOM_STATE)
    total_start = time.time()

    for ds_name, paths in datasets.items():
        print(f"\n{'='*60}")
        print(f"  Dataset: {ds_name}")
        print(f"{'='*60}")
        X, y = load_dataset(paths['train'])

        all_results[ds_name] = {
            'baseline': {c: {'acc': [], 'f1': [], 'params': []}
                         for c in classifiers},
            'relieff':  {c: {'acc': [], 'f1': [], 'params': []}
                         for c in classifiers},
        }
        fold_feature_scores = []
        fold_best_ks = []

        for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y)):

            print(f"\n  ── Fold {fold_idx+1}/{N_FOLDS} ──")
            X_tr, X_te = X[train_idx], X[test_idx]
            y_tr, y_te = y[train_idx], y[test_idx]

            # ── Phase 1 : Baseline ──────────────────────────────
            print("  [Baseline]")
            for clf_name, (clf, grid) in classifiers.items():
                t0          = time.time()
                acc, f1, params = evaluate_classifier(
                    clf, grid, X_tr, y_tr, X_te, y_te)
                elapsed     = time.time() - t0

                all_results[ds_name]['baseline'][clf_name]['acc'].append(acc)
                all_results[ds_name]['baseline'][clf_name]['f1'].append(f1)
                all_results[ds_name]['baseline'][clf_name]['params'].append(params)

                print(f"    {clf_name:15s}  acc={acc:.4f}  f1={f1:.4f}"
                      f"  ({elapsed:.1f}s)")

            # ── Phase 2 : ReliefF ───────────────────────────────
            print("  [ReliefF] ranking features …", end=" ", flush=True)
            t0 = time.time()
            _, importances, _ = apply_relieff(X_tr, y_tr, int(max(K_CANDIDATES)))
            print(f"{time.time()-t0:.1f}s")

            # Pick best k with a fast RF proxy in inner CV
            best_k, best_score = K_CANDIDATES[0], -np.inf
            for k in K_CANDIDATES:
                if k > X_tr.shape[1]:
                    continue
                top_idx  = np.argsort(importances)[::-1][:k]
                Xtr_k    = X_tr[:, top_idx]
                inner    = StratifiedKFold(n_splits=3, shuffle=True,
                                           random_state=0)
                scores   = []
                for itr, ite in inner.split(Xtr_k, y_tr):
                    rf = RandomForestClassifier(n_estimators=50,
                                                random_state=RANDOM_STATE,
                                                n_jobs=-1)
                    rf.fit(Xtr_k[itr], y_tr[itr])
                    scores.append(f1_score(y_tr[ite],
                                           rf.predict(Xtr_k[ite]),
                                           average='macro', zero_division=0))
                if np.mean(scores) > best_score:
                    best_score = np.mean(scores)
                    best_k     = k

            top_final = np.argsort(importances)[::-1][:best_k]
            X_tr_r, X_te_r    = X_tr[:, top_final], X_te[:, top_final]
            fold_feature_scores.append(importances)
            fold_best_ks.append(best_k)
            print(f"  [ReliefF] best_k={best_k}")

            #Phase 2: ReliefF Classifiers
            for clf_name, (clf, grid) in classifiers.items():
                t0 = time.time()
                acc, f1, params = evaluate_classifier(
                    clf, grid, X_tr_r, y_tr, X_te_r, y_te)
                elapsed     = time.time() - t0

                all_results[ds_name]['relieff'][clf_name]['acc'].append(acc)
                all_results[ds_name]['relieff'][clf_name]['f1'].append(f1)
                all_results[ds_name]['relieff'][clf_name]['params'].append(params)

                print(f"    {clf_name:15s}  acc={acc:.4f}  f1={f1:.4f}"
                      f"  ({elapsed:.1f}s)")

        relieff_scores[ds_name] = np.mean(fold_feature_scores, axis=0)
        best_k_per_ds[ds_name]  = int(np.median(fold_best_ks))

        # Save checkpoint after every dataset (safe if interrupted)
        with open(CHECKPOINT_FILE, 'wb') as f:
            pickle.dump({'all_results':    all_results,
                         'relieff_scores': relieff_scores, 
                         'best_k_per_ds': best_k_per_ds}, f)
        print(f"\n  ✅  Checkpoint saved after {ds_name}")

    elapsed_total = time.time() - total_start
    print(f"\n{'='*60}")
    print(f"  All experiments done in {elapsed_total/60:.1f} min")
    print(f"{'='*60}\n")
    return all_results, relieff_scores

all_results, relieff_scores, best_k_per_ds = training_models(DATASETS, CLASSIFIERS)


  Dataset: Dataset1
  Loaded: 9583 samples, 19 features, 4 classes

  ── Fold 1/10 ──
  [Baseline]


KeyboardInterrupt: 

In [ ]:
# Load checkpoint if re-running this cell independently
if 'best_k_per_ds' not in dir():
    with open(CHECKPOINT_FILE, 'rb') as f:
        ck = pickle.load(f)
    all_results    = ck['all_results']
    relieff_scores = ck['relieff_scores']
    best_k_per_ds  = ck['best_k_per_ds']

final_test_results = {}

for ds_name, paths in DATASETS.items():
    print(f'\n=== Final Test: {ds_name} ===')
    X_train, y_train = load_dataset(paths['train'])
    X_test,  y_test  = load_dataset(paths['test'])

    final_test_results[ds_name] = {'baseline': {}, 'relieff': {}}

    # Baseline
    for clf_name, (clf, grid) in CLASSIFIERS.items():
        acc, f1, params = evaluate_classifier(clf, grid, X_train, y_train, X_test, y_test)
        final_test_results[ds_name]['baseline'][clf_name] = {'acc': acc, 'f1': f1, 'params': params}
        print(f'  [Baseline] {clf_name:15s}  acc={acc:.4f}  f1={f1:.4f}')

    # ReliefF — use median best_k from CV folds
    best_k  = best_k_per_ds.get(ds_name, K_CANDIDATES[0])
    _, importances, _ = apply_relieff(X_train, y_train, int(max(K_CANDIDATES)))
    top_idx = np.argsort(importances)[::-1][:best_k]
    X_train_r, X_test_r = X_train[:, top_idx], X_test[:, top_idx]

    for clf_name, (clf, grid) in CLASSIFIERS.items():
        acc, f1, params = evaluate_classifier(clf, grid, X_train_r, y_train, X_test_r, y_test)
        final_test_results[ds_name]['relieff'][clf_name] = {'acc': acc, 'f1': f1, 'params': params}
        print(f'  [ReliefF]  {clf_name:15s}  acc={acc:.4f}  f1={f1:.4f}')

with open('final_test_results1-8.pkl', 'wb') as f:
    pickle.dump(final_test_results, f)
print('\n✅  final_test_results1-8.pkl saved.')
